In [1]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt


# ============================================================
# Two identical atoms with relatively strong dipole-dipole
# interaction.
#
# Global / frequency-resolved GKSL master equation.
#
# hbar = k_B = 1
# ============================================================


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

w0 = 1.0

# Strong compared with linewidth, but still small compared
# with optical transition frequency.
Omega12 = 0.03

Gamma0 = 0.002
Gamma12_0 = 0.0005

T = 0.5


# ============================================================
# Single-atom states and operators
# ============================================================

g = qt.basis(2, 0)
e = qt.basis(2, 1)

sm0 = g * e.dag()
sp0 = sm0.dag()
n0  = sp0 * sm0

I = qt.qeye(2)

sm1 = qt.tensor(sm0, I)
sm2 = qt.tensor(I, sm0)

sp1 = sm1.dag()
sp2 = sm2.dag()

n1 = qt.tensor(n0, I)
n2 = qt.tensor(I, n0)


# ============================================================
# Product and Dicke states
# ============================================================

gg = qt.tensor(g, g)
eg = qt.tensor(e, g)
ge = qt.tensor(g, e)
ee = qt.tensor(e, e)

s = (eg + ge).unit()
a = (eg - ge).unit()

Pgg = gg.proj()
Ps  = s.proj()
Pa  = a.proj()
Pee = ee.proj()


# ============================================================
# Hamiltonian
#
# H = w0 (n1+n2)
#     + Omega12 (sigma1+ sigma2- + sigma2+ sigma1-)
# ============================================================

H = (
    w0 * (n1 + n2)
    + Omega12 * (sp1 * sm2 + sp2 * sm1)
)


# ============================================================
# Energies
# ============================================================

Eg = 0.0
Es = w0 + Omega12
Ea = w0 - Omega12
Ee = 2.0 * w0

w_plus  = w0 + Omega12
w_minus = w0 - Omega12


print("Dicke energies:")
print(f"Eg = {Eg:.6f}")
print(f"Ea = {Ea:.6f}")
print(f"Es = {Es:.6f}")
print(f"Ee = {Ee:.6f}")

print("\nTransition frequencies:")
print(f"omega_plus  = {w_plus:.6f}")
print(f"omega_minus = {w_minus:.6f}")

print(
    "\nResolved-splitting ratio 2 Omega / Gamma0 =",
    2.0 * abs(Omega12) / Gamma0
)


# ============================================================
# Symmetric and antisymmetric bath operators
#
# These diagonalize the spatial decay matrix
#
# Gamma =
#
#   Gamma0    Gamma12
#   Gamma12   Gamma0
#
# ============================================================

S_sym  = (sm1 + sm2) / np.sqrt(2.0)
S_anti = (sm1 - sm2) / np.sqrt(2.0)


# ============================================================
# Frequency components
#
# Instead of using S_sym and S_anti as single collapse
# operators, split them according to the true Bohr frequencies.
#
# Symmetric:
#
# ee -> s : omega_minus
# s  -> gg: omega_plus
#
# Antisymmetric:
#
# ee -> a : omega_plus
# a  -> gg: omega_minus
# ============================================================

A_es = Ps  * S_sym  * Pee
A_sg = Pgg * S_sym  * Ps

A_ea = Pa  * S_anti * Pee
A_ag = Pgg * S_anti * Pa


# Check their norms / matrix elements
print("\nTransition operator norms:")
print("ee -> s :", A_es.norm())
print("s  -> g :", A_sg.norm())
print("ee -> a :", A_ea.norm())
print("a  -> g :", A_ag.norm())


# ============================================================
# Thermal occupation
# ============================================================

def nbar(w):
    if T <= 0:
        return 0.0

    x = w / T

    if x > 700:
        return 0.0

    return 1.0 / np.expm1(x)


# ============================================================
# Frequency-dependent decay rates
#
# Simple free-space scaling:
#
# Gamma(w) ~ w^3
#
# Here Gamma12/Gamma0 is assumed approximately constant over
# the small frequency interval w0 +/- Omega12.
#
# If desired, replace these functions by the full
# frequency-dependent Lehmberg expressions.
# ============================================================

def Gamma_single(w):

    if w <= 0:
        return 0.0

    return Gamma0 * (w / w0)**3


def Gamma_cross(w):

    if w <= 0:
        return 0.0

    return Gamma12_0 * (w / w0)**3


def Gamma_sym(w):

    value = Gamma_single(w) + Gamma_cross(w)

    if value < 0:
        raise ValueError("Negative symmetric decay rate")

    return value


def Gamma_anti(w):

    value = Gamma_single(w) - Gamma_cross(w)

    if value < 0:
        raise ValueError("Negative antisymmetric decay rate")

    return value


# ============================================================
# Construct one downward and one upward collapse operator
# for each physical transition.
#
# Detailed balance:
#
# Gamma_up / Gamma_down
#
#       = n / (n+1)
#
#       = exp(-w/T)
#
# ============================================================

transitions = [

    # name       A_down   frequency   bath rate

    ("e -> s",   A_es,    w_minus,    Gamma_sym),
    ("s -> g",   A_sg,    w_plus,     Gamma_sym),

    ("e -> a",   A_ea,    w_plus,     Gamma_anti),
    ("a -> g",   A_ag,    w_minus,    Gamma_anti),
]


c_ops = []

print("\nTransition rates")
print("----------------")

for name, A, w, Gamma_channel in transitions:

    n = nbar(w)
    gamma = Gamma_channel(w)

    rate_down = gamma * (n + 1.0)
    rate_up   = gamma * n

    C_down = np.sqrt(rate_down) * A
    C_up   = np.sqrt(rate_up) * A.dag()

    c_ops.append(C_down)

    # At T = 0 this rate vanishes, so there is no need
    # to add a zero operator.
    if rate_up > 1e-15:
        c_ops.append(C_up)

    print(
        f"{name:7s}   "
        f"w={w:.6f}   "
        f"n={n:.6f}   "
        f"down={rate_down:.8f}   "
        f"up={rate_up:.8f}"
    )

    if rate_down > 0:
        print(
            "             up/down =",
            rate_up / rate_down,
            "  exp(-w/T) =",
            np.exp(-w / T)
        )


print("\nNumber of collapse operators =", len(c_ops))


# ============================================================
# Steady state
# ============================================================

rho_ss = qt.steadystate(
    H,
    c_ops,
    method="direct"
)


# ============================================================
# Exact Gibbs state
#
# rho_G = exp(-H/T) / Z
# ============================================================

if T > 0:

    rho_G = (-H / T).expm()
    rho_G = rho_G / rho_G.tr()

    trace_distance = (
        0.5 * (rho_ss - rho_G).norm("tr")
    )

    print(
        "\nTrace distance between GKSL steady state"
        " and exact Gibbs state:"
    )

    print(trace_distance)


# ============================================================
# Populations in Dicke basis
# ============================================================

print("\nSteady-state Dicke populations")
print("-------------------------------")

print(
    "Pg =",
    np.real(qt.expect(Pgg, rho_ss))
)

print(
    "Ps =",
    np.real(qt.expect(Ps, rho_ss))
)

print(
    "Pa =",
    np.real(qt.expect(Pa, rho_ss))
)

print(
    "Pe =",
    np.real(qt.expect(Pee, rho_ss))
)


# ============================================================
# Positivity check
# ============================================================

rho_eigs = np.linalg.eigvalsh(
    rho_ss.full()
)

print("\nEigenvalues of rho_ss:")
print(rho_eigs)

print(
    "minimum eigenvalue =",
    rho_eigs.min()
)


# ============================================================
# Example time evolution
#
# Start from |ee>.
# ============================================================

rho0 = Pee

tlist = np.linspace(
    0.0,
    3000.0,
    1501
)

e_ops = {
    "gg": Pgg,
    "s":  Ps,
    "a":  Pa,
    "ee": Pee
}

result = qt.mesolve(
    H,
    rho0,
    tlist,
    c_ops=c_ops,
    e_ops=e_ops,
    options={
        "progress_bar": ""
    }
)


# ============================================================
# Plot
# ============================================================

plt.figure()

for name in ["gg", "s", "a", "ee"]:

    plt.plot(
        tlist,
        result.expect[name],
        label=name
    )

plt.xlabel("time")
plt.ylabel("population")
plt.legend()
plt.tight_layout()
plt.show()

Dicke energies:
Eg = 0.000000
Ea = 0.970000
Es = 1.030000
Ee = 2.000000

Transition frequencies:
omega_plus  = 1.030000
omega_minus = 0.970000

Resolved-splitting ratio 2 Omega / Gamma0 = 30.0

Transition operator norms:
ee -> s : 0.9999999999999998
s  -> g : 0.9999999999999998
ee -> a : 0.9999999999999997
a  -> g : 0.9999999999999998

Transition rates
----------------
e -> s    w=0.970000   n=0.167820   down=0.00266460   up=0.00038291
             up/down = 0.14370394977770293   exp(-w/T) = 0.14370394977770293
s -> g    w=1.030000   n=0.146071   down=0.00313086   up=0.00039904
             up/down = 0.12745396989482075   exp(-w/T) = 0.12745396989482075
e -> a    w=1.030000   n=0.146071   down=0.00187851   up=0.00023942
             up/down = 0.12745396989482075   exp(-w/T) = 0.12745396989482075
a -> g    w=0.970000   n=0.167820   down=0.00159876   up=0.00022975
             up/down = 0.14370394977770296   exp(-w/T) = 0.14370394977770293

Number of collapse operators = 8

Trace distanc

TypeError: list indices must be integers or slices, not str

<Figure size 640x480 with 0 Axes>